In [2]:
import xml.etree.ElementTree as ET
import html
import re
import copy
from trees import visualize_element_tree
from graphviz import Digraph


class PMC_article:
    """A class defining attributes and methods for parsing a PMC article XML file"""
    
    def __init__(self, pmc_xml_path):
        
        self.path = pmc_xml_path
        self._root = None
        self.article_title = None
        self.journal_name = None
        self.abstract_full_text = None
        self.abstract_plain_text = None
        self.abstract_sections = None
        self.abstract_string=None
        
        self._parse_xml()
        self._extract_article_title()
        self._extract_journal_name()
        self._extract_abstract()


    def _parse_xml(self):
        """Parses XML file and stores the root Element."""
        try:
            tree = ET.parse(self.path)
            self._root = tree.getroot()
        except Exception as e:
            raise RuntimeError(f"Failed to parse XML file: {self.path}") from e

    def print_tree_structure(self, element=None, level=1):
        """Prints tree structure"""
        if element is None:
            element = self._root
        for child in element:
            print(" "*level, "-", child.tag)
            self.print_tree_structure(element=child,level=level+1)


    def _extract_article_title(self):
        """Extract title of the article from the parsed XML."""
        article_element = self._root.find(".//article-title") ## finds first occurence of the article title
        if article_element is not None:
            self.article_title = " ".join( t.strip() for t in article_element.itertext() if t.strip())
    
    @staticmethod
    def replace_styling_tags(text, replaces={"<sub>": "_", "<sup>": "^"}):
        """Replace styling tags (e.g., sub/sup) while preserving meaning."""
        if not text:
            return ""

        for tag, replacement in replaces.items():
            # Match opening tag with optional attributes
            open_tag_pattern = r"\s*" + tag[:-1] + r"\b[^>]*" + tag[-1]
            # Match corresponding closing tag
            close_tag_pattern = tag[0] + r"/" + tag[1:]

            # Replace opening tag
            text = re.sub(open_tag_pattern, replacement, text, flags=re.IGNORECASE)
            # Remove closing tag
            text = re.sub(close_tag_pattern, "", text, flags=re.IGNORECASE)

        return text
        
    @staticmethod
    def clean_text(text):
        """Clean and normalize text content."""
        if not text:
            return ""
        
        # Decode HTML entities
        text = html.unescape(text)
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text)
        
        # Strip leading/trailing whitespace
        text = text.strip()
        
        return text
        


    def _extract_journal_name(self):
        """Extract name of the journal given root"""
        journal_element = self._root.find(".//journal-title")
        if journal_element is not None and journal_element.text:
            self.journal_name = " ".join( t.strip() for t in journal_element.itertext() if t.strip())


    def _extract_abstract(self):
        """Extract abstract of article given root"""
        abstract = self._root.find(".//abstract")

        if abstract is not None:
            raw_abstract_xml = ET.tostring(abstract, encoding="unicode", method="xml")
            self.abstract_string=raw_abstract_xml
            styled_xml = self.replace_styling_tags(raw_abstract_xml)
            abstract_element = ET.fromstring(styled_xml)
            abstract_text = "".join(abstract_element.itertext())
            self.abstract_full_text = self.clean_text(abstract_text)

        ## Extracting plain abstract text
        abstract_plain_text=[]
        for child in abstract_element.iter():
            if child.tag == 'title':
                continue
            if child.text:
                abstract_plain_text.append(child.text)

            if child.tail:
                abstract_plain_text.append(child.tail)
        self.abstract_plain_text=self.clean_text("".join(abstract_plain_text).strip()) 

        abstract_structured_text=[]
        
        for element in abstract_element.iter():
            if element.tag == "title":
                abstract_structured_text.append(get_smart_text(element))
                abstract_structured_text.append(r": \n")
            elif element.tag="p":
               abstract_structured_text.append(get_smart_text(element))
                
                
        
        ## Extracting abstract sections
        abstract_sections = []
        
        for sec in abstract_element.findall(".//sec"):
            section = {}
        
            # --------
            # Section title (if present)
            # --------
            title_el = sec.find("title")
            if title_el is not None:
                section["title"] = self.clean_text(" ".join(title_el.itertext()))
            else:
                section["title"] = None
        
            # --------
            # Section body text (all paragraphs inside this section)
            # --------
            paragraphs = []
            for p in sec.findall(".//p"):
                paragraphs.append(" ".join(p.itertext()))
        
            section["text"] = self.clean_text(" ".join(paragraphs))
        
            abstract_sections.append(section)
        
        self.abstract_sections = abstract_sections

    

In [3]:
article1=PMC_article("./data/entrez_download_PMCID=7067710.xml")
article1.abstract_sections

Introduction
A fixed-dose combination (FDC) of ibuprofen and acetaminophen has been developed that provides greater analgesic efficacy than either agent alone at the same doses without increasing the risk for adverse events.
Methods
We report three clinical phase I studies designed to assess the pharmacokinetics (PK) of the FDC of ibuprofen/acetaminophen 250/500 mg (administered as two tablets of ibuprofen 125 mg/acetaminophen 250 mg) in comparison with its individual components administered alone or together, and to determine the effect of food on the PK of the FDC. Two studies in healthy adults aged 18–55 years used a crossover design in which subjects received a single dose of each treatment with a 2-day washout period between each. In the third study, the bioavailability of ibuprofen and acetaminophen from a single oral dose of the FDC was assessed in healthy adolescents aged 12–17 years, inclusive.
Results
A total of 35 and 46 subjects were enrolled in the two adult studies, respe

[{'title': 'Introduction',
  'text': 'A fixed-dose combination (FDC) of ibuprofen and acetaminophen has been developed that provides greater analgesic efficacy than either agent alone at the same doses without increasing the risk for adverse events.'},
 {'title': 'Methods',
  'text': 'We report three clinical phase I studies designed to assess the pharmacokinetics (PK) of the FDC of ibuprofen/acetaminophen 250/500 mg (administered as two tablets of ibuprofen 125 mg/acetaminophen 250 mg) in comparison with its individual components administered alone or together, and to determine the effect of food on the PK of the FDC. Two studies in healthy adults aged 18–55 years used a crossover design in which subjects received a single dose of each treatment with a 2-day washout period between each. In the third study, the bioavailability of ibuprofen and acetaminophen from a single oral dose of the FDC was assessed in healthy adolescents aged 12–17 years, inclusive.'},
 {'title': 'Results',
  'te

In [66]:
article1.abstract_plain_text

'A fixed-dose combination (FDC) of ibuprofen and acetaminophen has been developed that provides greater analgesic efficacy than either agent alone at the same doses without increasing the risk for adverse events.We report three clinical phase I studies designed to assess the pharmacokinetics (PK) of the FDC of ibuprofen/acetaminophen 250/500 mg (administered as two tablets of ibuprofen 125 mg/acetaminophen 250 mg) in comparison with its individual components administered alone or together, and to determine the effect of food on the PK of the FDC. Two studies in healthy adults aged 18–55 years used a crossover design in which subjects received a single dose of each treatment with a 2-day washout period between each. In the third study, the bioavailability of ibuprofen and acetaminophen from a single oral dose of the FDC was assessed in healthy adolescents aged 12–17 years, inclusive.A total of 35 and 46 subjects were enrolled in the two adult studies, respectively, and 21 were enrolled 

In [67]:
article1.abstract_full_text

'IntroductionA fixed-dose combination (FDC) of ibuprofen and acetaminophen has been developed that provides greater analgesic efficacy than either agent alone at the same doses without increasing the risk for adverse events.MethodsWe report three clinical phase I studies designed to assess the pharmacokinetics (PK) of the FDC of ibuprofen/acetaminophen 250/500 mg (administered as two tablets of ibuprofen 125 mg/acetaminophen 250 mg) in comparison with its individual components administered alone or together, and to determine the effect of food on the PK of the FDC. Two studies in healthy adults aged 18–55 years used a crossover design in which subjects received a single dose of each treatment with a 2-day washout period between each. In the third study, the bioavailability of ibuprofen and acetaminophen from a single oral dose of the FDC was assessed in healthy adolescents aged 12–17 years, inclusive.ResultsA total of 35 and 46 subjects were enrolled in the two adult studies, respectiv

In [116]:
from inspect import cleandoc

In [ ]:
cleandoc